# Task 2 — Notebook 5: Feature Engineering

**Objective:**
- Build the features decided on during EDA (Notebook 4)
- Handle missing values, encode categories, and scale numeric features
- Use **only information available at prediction time** — no future/leakage columns
- Fit every transformation on the **training split only**, then apply it to validation and test
- Save every fitted object (imputer, scaler, encoder) — not just the output table
- **Artifact:** the final feature tables (train/validation/test), the fitted transformers, and the feature list

> This notebook reads `train.parquet`, `validation.parquet`, and `test.parquet` from Notebook 3. Any fitting
> (imputer, scaler, encoder) happens **on train only**; validation and test are transformed using the same
> fitted objects — never refit.

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import os
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

os.makedirs("artifacts/transformers", exist_ok=True)

train_df = pd.read_parquet("artifacts/train.parquet")
val_df = pd.read_parquet("artifacts/validation.parquet")
test_df = pd.read_parquet("artifacts/test.parquet")

date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for df in (train_df, val_df, test_df):
    for col in date_cols:
        if not pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = pd.to_datetime(df[col])

print("Train:", train_df.shape, "| Val:", val_df.shape, "| Test:", test_df.shape)

Train: (67529, 21) | Val: (14470, 21) | Test: (14471, 21)


## 1. Recap: decisions from Notebook 4 (EDA)

From `artifacts/eda_findings.md`:

- **Strong candidate features:** `customer_state`, `primary_seller_state`, `same_state`, `purchase_month`
  (seasonality), `total_price`, `total_freight`
- **Needs preprocessing:** right-skewed numeric columns (log-transform), a handful of missing values
  (median imputation), outliers kept as-is for now
- **Excluded:** `order_status` (zero variance), `customer_city` (high cardinality, low value), `delivery_days`
  and anything derived from `order_delivered_customer_date` (leakage)
- **Class imbalance:** 91.89% on-time / 8.11% late — relevant for Notebook 6, not this notebook

## 2. New decision: what "prediction time" actually means here

Before building features, we need to pin down **exactly when** the model makes its prediction — this decides
which columns are legitimate features vs. leakage.

**Decision:** the model predicts **right when the order is placed** (at `order_purchase_timestamp`). At that
moment we know: the customer, the items in the cart, the payment method chosen, the seller(s) assigned, and
the estimated delivery date shown at checkout. We do **not** yet know `order_approved_at` or
`order_delivered_carrier_date` — those are events that happen *after* the order is placed, so they are
**excluded from features entirely** (not just imputed — dropped as columns). This also resolves 15 of the 18
missing values found in Notebook 4 (`order_approved_at`: 14, `order_delivered_carrier_date`: 1) automatically,
since we're not using those columns at all.

The remaining missing values (`n_payments`, `total_payment_value`, `max_installments` — 1 each) are legitimate
checkout-time information, so those will be imputed rather than dropped.

## 3. Build seller-geography features (`primary_seller_state`, `same_state`)

Same logic used for exploration in Notebook 4's geography section — computed here for **all three splits**
independently (this is just a join, not a fitted transformation, so no train/val/test asymmetry issue).

In [3]:
from sqlalchemy import create_engine

DB_USER = "olist_user"
DB_PASS = "olist_pass"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "olist_db"

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

order_items_raw = pd.read_sql_table("order_items", engine)
sellers_raw = pd.read_sql_table("sellers", engine)
items_sellers_all = order_items_raw.merge(sellers_raw, on="seller_id", how="left")

def mode_or_none(s):
    m = s.mode()
    return m.iloc[0] if not m.empty else None

seller_state_lookup = (
    items_sellers_all.groupby("order_id")["seller_state"]
    .agg(mode_or_none)
    .rename("primary_seller_state")
    .reset_index()
)

def add_seller_geo_features(df):
    df = df.merge(seller_state_lookup, on="order_id", how="left")
    df["same_state"] = (df["customer_state"] == df["primary_seller_state"]).astype(int)
    return df

train_df = add_seller_geo_features(train_df)
val_df = add_seller_geo_features(val_df)
test_df = add_seller_geo_features(test_df)

for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"{name}: missing primary_seller_state = {df['primary_seller_state'].isna().sum()}")

Train: missing primary_seller_state = 0
Val: missing primary_seller_state = 0
Test: missing primary_seller_state = 0


## 4. Build date-derived features

- `estimated_days`: purchase → estimated delivery (available at checkout, safe)
- `purchase_month_num` / `purchase_weekday_num`: encoded **cyclically** (sin/cos) instead of one-hot on the
  raw `YYYY-MM` string — this generalizes to months/years the model hasn't seen in training (important since
  our split is time-based and the test set is entirely future months)
- `is_peak_season`: flag for Nov/Feb/Mar, based on the clear seasonal spikes found in the EDA

In [4]:
def add_date_features(df):
    df = df.copy()
    df["estimated_days"] = (
        df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
    ).dt.days

    month = df["order_purchase_timestamp"].dt.month
    weekday = df["order_purchase_timestamp"].dt.dayofweek

    df["purchase_month_sin"] = np.sin(2 * np.pi * month / 12)
    df["purchase_month_cos"] = np.cos(2 * np.pi * month / 12)
    df["purchase_weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
    df["purchase_weekday_cos"] = np.cos(2 * np.pi * weekday / 7)

    df["is_peak_season"] = month.isin([11, 2, 3]).astype(int)
    return df

train_df = add_date_features(train_df)
val_df = add_date_features(val_df)
test_df = add_date_features(test_df)

train_df[["estimated_days", "purchase_month_sin", "purchase_month_cos", "is_peak_season"]].head()

,estimated_days,purchase_month_sin,purchase_month_cos,is_peak_season
0,18,-1.000000,-1.836970e-16,0
1,23,-0.866025,5.000000e-01,0
2,34,-0.866025,5.000000e-01,0
3,56,-0.866025,5.000000e-01,0
4,50,-0.866025,5.000000e-01,0


## 5. Select the feature columns

Assemble the raw (pre-processing) feature set based on the EDA decisions:
- Drop leakage columns, IDs, `order_status` (zero variance), `customer_city` (high cardinality)
- Keep the numeric order/payment aggregates, the new date features, and the geography features

In [5]:
numeric_features = [
    "n_items", "n_distinct_products", "n_distinct_sellers",
    "total_price", "total_freight", "n_payments",
    "total_payment_value", "max_installments", "estimated_days",
]

cyclical_features = [
    "purchase_month_sin", "purchase_month_cos",
    "purchase_weekday_sin", "purchase_weekday_cos",
]

binary_features = ["same_state", "is_peak_season"]

categorical_features = ["customer_state", "primary_seller_state"]

label_col = "is_late"

all_raw_columns = numeric_features + cyclical_features + binary_features + categorical_features + [label_col]

train_raw = train_df[all_raw_columns].copy()
val_raw = val_df[all_raw_columns].copy()
test_raw = test_df[all_raw_columns].copy()

print(train_raw.shape, val_raw.shape, test_raw.shape)

(67529, 18) (14470, 18) (14471, 18)


## 6. Handle missing values

Only `n_payments`, `total_payment_value`, `max_installments` have missing values left (1 each, from the
Notebook 1 left join). We fit a **median imputer on train only**, then apply the same fitted values to
validation and test — never refit on new data.

In [6]:
num_imputer = SimpleImputer(strategy="median")
train_raw[numeric_features] = num_imputer.fit_transform(train_raw[numeric_features])
val_raw[numeric_features] = num_imputer.transform(val_raw[numeric_features])
test_raw[numeric_features] = num_imputer.transform(test_raw[numeric_features])

joblib.dump(num_imputer, "artifacts/transformers/numeric_imputer.joblib")

print("Remaining missing values after imputation:")
print("Train:", train_raw[numeric_features].isna().sum().sum())
print("Val:  ", val_raw[numeric_features].isna().sum().sum())
print("Test: ", test_raw[numeric_features].isna().sum().sum())

Remaining missing values after imputation:
Train: 0
Val:   0
Test:  0


In [7]:
# categorical features: fill any missing seller-state matches with a placeholder category
# (fit not needed here — this is a fixed rule, not something learned from the data)
for df in (train_raw, val_raw, test_raw):
    df["customer_state"] = df["customer_state"].fillna("UNKNOWN")
    df["primary_seller_state"] = df["primary_seller_state"].fillna("UNKNOWN")

## 7. Log-transform skewed numeric features

From the EDA, `n_payments`, `total_price`, `n_distinct_sellers`, `total_payment_value`, and `total_freight`
are strongly right-skewed. We apply `log1p` (handles zero values safely) — this is a fixed, deterministic
transformation, not something fitted on the data, so it's applied identically to all three splits.

In [8]:
skewed_cols = ["n_payments", "total_price", "n_distinct_sellers", "total_payment_value", "total_freight"]

for df in (train_raw, val_raw, test_raw):
    for col in skewed_cols:
        df[col] = np.log1p(df[col])

train_raw[skewed_cols].skew()

n_payments             8.958158
total_price            0.282638
n_distinct_sellers     9.700384
total_payment_value    0.549224
total_freight          1.179743
dtype: float64

## 8. Scale numeric features

We fit a `StandardScaler` on the training numeric columns (after log-transform), and apply the same fitted
scaler to validation and test. This matters for linear/distance-based models; it's harmless for tree-based
models too, so we keep it generic rather than assuming the model choice from Notebook 6 upfront.

In [9]:
scaler = StandardScaler()
train_raw[numeric_features] = scaler.fit_transform(train_raw[numeric_features])
val_raw[numeric_features] = scaler.transform(val_raw[numeric_features])
test_raw[numeric_features] = scaler.transform(test_raw[numeric_features])

joblib.dump(scaler, "artifacts/transformers/numeric_scaler.joblib")

train_raw[numeric_features].describe().T[["mean", "std"]]

,mean,std
n_items,7.144464e-17,1.000007
n_distinct_products,-2.735730e-17,1.000007
n_distinct_sellers,5.763973e-16,1.000007
total_price,-5.441998e-16,1.000007
total_freight,1.032843e-15,1.000007
n_payments,-8.312410e-16,1.000007
total_payment_value,6.477367e-16,1.000007
max_installments,5.892342e-18,1.000007
estimated_days,-4.503432e-17,1.000007


## 9. Encode categorical features

`OneHotEncoder` fit on train only, with `handle_unknown="ignore"` — so if validation/test contain a
`customer_state` or `primary_seller_state` value never seen in training, it's encoded as all-zeros instead of
crashing the pipeline (this matters in production, where new states/regions could appear).

In [10]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(train_raw[categorical_features])

encoded_cols = encoder.get_feature_names_out(categorical_features)

train_encoded = pd.DataFrame(
    encoder.transform(train_raw[categorical_features]), columns=encoded_cols, index=train_raw.index
)
val_encoded = pd.DataFrame(
    encoder.transform(val_raw[categorical_features]), columns=encoded_cols, index=val_raw.index
)
test_encoded = pd.DataFrame(
    encoder.transform(test_raw[categorical_features]), columns=encoded_cols, index=test_raw.index
)

joblib.dump(encoder, "artifacts/transformers/categorical_encoder.joblib")

print("Encoded columns:", len(encoded_cols))
train_encoded.head()

Encoded columns: 49


,customer_state_AC,customer_state_AL,customer_state_AM,customer_state_AP,customer_state_BA,customer_state_CE,customer_state_DF,customer_state_ES,customer_state_GO,customer_state_MA,...,primary_seller_state_PE,primary_seller_state_PI,primary_seller_state_PR,primary_seller_state_RJ,primary_seller_state_RN,primary_seller_state_RO,primary_seller_state_RS,primary_seller_state_SC,primary_seller_state_SE,primary_seller_state_SP
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 10. Assemble the final feature tables

In [11]:
def assemble(df_raw, df_encoded):
    return pd.concat(
        [df_raw[numeric_features + cyclical_features + binary_features], df_encoded, df_raw[[label_col]]],
        axis=1
    )

train_features = assemble(train_raw, train_encoded)
val_features = assemble(val_raw, val_encoded)
test_features = assemble(test_raw, test_encoded)

print("Train features:", train_features.shape)
print("Val features:  ", val_features.shape)
print("Test features: ", test_features.shape)
train_features.head()

Train features: (67529, 65)
Val features:   (14470, 65)
Test features:  (14471, 65)


,n_items,n_distinct_products,n_distinct_sellers,total_price,total_freight,n_payments,total_payment_value,max_installments,estimated_days,purchase_month_sin,...,primary_seller_state_PI,primary_seller_state_PR,primary_seller_state_RJ,primary_seller_state_RN,primary_seller_state_RO,primary_seller_state_RS,primary_seller_state_SC,primary_seller_state_SE,primary_seller_state_SP,is_late
0,3.430214,-0.16968,-0.106476,0.501859,-1.451697,-0.159034,-0.050824,-0.353720,-0.777493,-1.000000,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,-0.262302,-0.16968,-0.106476,-1.115623,-0.350569,-0.159034,-1.076835,-0.716679,-0.153694,-0.866025,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,-0.262302,-0.16968,-0.106476,-1.442704,-0.164892,-0.159034,-1.261985,-0.716679,1.218665,-0.866025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
3,-0.262302,-0.16968,-0.106476,-0.904586,-0.159463,-0.159034,-0.871149,-0.716679,3.963381,-0.866025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
4,-0.262302,-0.16968,-0.106476,0.373622,-0.605132,-0.159034,0.257413,1.098117,3.214822,-0.866025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0


## 11. Sanity checks

Before saving, we confirm:
1. No leakage columns present (`order_delivered_customer_date`, `order_approved_at`, `order_delivered_carrier_date`)
2. No missing values remaining in any split
3. Train, validation, and test have the **exact same columns** (critical — the model expects identical shape)

In [12]:
leakage_cols = ["order_delivered_customer_date", "order_approved_at", "order_delivered_carrier_date", "delivery_days"]
for col in leakage_cols:
    assert col not in train_features.columns, f"Leakage column {col} found in features!"

for name, df in [("Train", train_features), ("Val", val_features), ("Test", test_features)]:
    n_missing = df.isna().sum().sum()
    assert n_missing == 0, f"{name} has {n_missing} missing values!"
    print(f"{name}: no missing values.")

assert list(train_features.columns) == list(val_features.columns) == list(test_features.columns), \
    "Column mismatch between splits!"
print("Train/Val/Test have identical columns.")

Train: no missing values.
Val: no missing values.
Test: no missing values.
Train/Val/Test have identical columns.


## 12. Save artifacts

- Final feature tables (train/validation/test)
- The fitted transformers (imputer, scaler, encoder) — required so production can apply the **exact same**
  transformations to new data, without ever refitting
- The feature list, for reference in Notebook 6 and beyond

In [13]:
train_features.to_parquet("artifacts/train_features.parquet", index=False)
val_features.to_parquet("artifacts/validation_features.parquet", index=False)
test_features.to_parquet("artifacts/test_features.parquet", index=False)

feature_list = {
    "numeric_features": numeric_features,
    "cyclical_features": cyclical_features,
    "binary_features": binary_features,
    "categorical_features_original": categorical_features,
    "categorical_features_encoded": list(encoded_cols),
    "label": label_col,
}
with open("artifacts/feature_list.json", "w", encoding="utf-8") as f:
    json.dump(feature_list, f, indent=2)

print("Saved:")
print(" - artifacts/train_features.parquet", train_features.shape)
print(" - artifacts/validation_features.parquet", val_features.shape)
print(" - artifacts/test_features.parquet", test_features.shape)
print(" - artifacts/transformers/numeric_imputer.joblib")
print(" - artifacts/transformers/numeric_scaler.joblib")
print(" - artifacts/transformers/categorical_encoder.joblib")
print(" - artifacts/feature_list.json")

Saved:
 - artifacts/train_features.parquet (67529, 65)
 - artifacts/validation_features.parquet (14470, 65)
 - artifacts/test_features.parquet (14471, 65)
 - artifacts/transformers/numeric_imputer.joblib
 - artifacts/transformers/numeric_scaler.joblib
 - artifacts/transformers/categorical_encoder.joblib
 - artifacts/feature_list.json


## ✅ Done when
- Feature tables saved for train, validation, and test — same columns, no missing values
- All fitted objects (imputer, scaler, encoder) saved separately — ready to be reused in production without refitting
- No leakage columns present anywhere in the final feature tables
- `feature_list.json` documents exactly which columns go into the model in Notebook 6